In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [5]:
df = pd.read_csv("../data/sales_data.csv")

In [6]:
# Inspect columns and types
print(df.columns)
print(df.dtypes)
print(df.head())

Index(['customer_id', 'store_name', 'transaction_date', 'aisle',
       'product_name', 'quantity', 'unit_price', 'total_amount',
       'discount_amount', 'final_amount', 'loyalty_points'],
      dtype='object')
customer_id           int64
store_name           object
transaction_date     object
aisle                object
product_name         object
quantity            float64
unit_price          float64
total_amount        float64
discount_amount     float64
final_amount        float64
loyalty_points        int64
dtype: object
   customer_id         store_name transaction_date           aisle  \
0         2824  GreenGrocer Plaza       2023-08-26         Produce   
1         5506   ValuePlus Market       2024-02-13           Dairy   
2         4657   ValuePlus Market       2023-11-23          Bakery   
3         2679  SuperSave Central       2025-01-13  Snacks & Candy   
4         9935  GreenGrocer Plaza       2023-10-13    Canned Goods   

   product_name  quantity  unit_price  total

In [7]:
# 3️⃣ Clean Data

# Strip column names and lowercase
df.columns = df.columns.str.strip().str.lower()

# Ensure transaction_date is datetime
df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')
df = df.dropna(subset=['transaction_date'])

# Ensure numeric columns are numeric
numeric_cols = ['quantity', 'unit_price', 'total_amount', 'discount_amount', 'final_amount', 'loyalty_points']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    
df = df.dropna(subset=['quantity', 'unit_price'])

# 4️⃣ Create Promotion Feature
# 1 if discount_amount > 0, else 0
df['promotion'] = df['discount_amount'].apply(lambda x: 1 if x > 0 else 0)

# 5️⃣ Feature Engineering

# Time-based features
df['day_of_week'] = df['transaction_date'].dt.dayofweek  # 0 = Monday
df['month'] = df['transaction_date'].dt.month
df['quarter'] = df['transaction_date'].dt.quarter

# Lag features (previous day sales per product)
df = df.sort_values(by=['product_name','transaction_date'])
df['lag_1'] = df.groupby('product_name')['quantity'].shift(1)
df['lag_7'] = df.groupby('product_name')['quantity'].shift(7)
df['rolling_7'] = df.groupby('product_name')['quantity'].rolling(window=7).mean().reset_index(0,drop=True)

# Fill NaNs in lag features with 0
df[['lag_1','lag_7','rolling_7']] = df[['lag_1','lag_7','rolling_7']].fillna(0)

# Encode categorical features
# product_name and aisle
le_product = LabelEncoder()
df['product_encoded'] = le_product.fit_transform(df['product_name'])

le_aisle = LabelEncoder()
df['aisle_encoded'] = le_aisle.fit_transform(df['aisle'])

# 6️⃣ Final Feature Selection
features = [
    'unit_price', 'promotion', 'day_of_week', 'month', 'quarter',
    'lag_1','lag_7','rolling_7','product_encoded','aisle_encoded'
]
target = 'quantity'

X = df[features]
y = df[target]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

# 7️⃣ Train/Test Split
# We'll use 80% train, 20% test
# For time series, do not shuffle
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

# 8️⃣ Save Prepared Data (Optional)
X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("Data preparation complete!")

Features shape: (1980, 10)
Target shape: (1980,)
Train shape: (1584, 10) (1584,)
Test shape: (396, 10) (396,)
Data preparation complete!
